# 04 — Signal Quality

Walks `SignalTradePerformance` method-by-method, then composes them all into
the standalone HTML tearsheet at the end.

**Pipeline**
```
YahooFinanceProvider  →  TrendSignal  →  BarBacktest  →  SignalTradePerformance       (iid bets)
      (bars)            (signal cols)   (bt_result)    │                              ▲
                                                       │                         this notebook
                                                       └─ SignalAllocationPerformance (capital deployed)
```

**Sections**

1. **Pipeline & setup** — `bars → signal_df → bt_result → trade_perf`. Each
   section below defines its own intermediate (`trades_df`, `ts_net`/`ts_con`,
   `paths`) and threads it forward via the `trade_stats=` kwarg, so nothing
   recomputes the per-trade groupby.
2. **`trade_stats()`** — defines `trades_df`. Canonical per-trade scalar table
   (one row per entry-to-exit cycle). Every other method derives from this.
3. **`trade_summary(method)`** — defines `ts_net` / `ts_con`. Per-symbol
   aggregation: win rate, expectancy, profit factor, skew, intra-trade DD.
4. **`quality_flags(row)`** — uses `ts_net` / `ts_con`. Warning flags from one
   row of a `trade_summary`. Cross-fill flags fire when non-net is compared to net.
5. **`d5_stats(symbol, method)`** — uses `trades_df`. Entry-timing breakdown:
   how often does a trade work by bar 5, and how often does the tail add return?
6. **`quality_table(methods)`** — uses `trades_df`. The joined view:
   trade_summary + d5_stats + quality_flags, indexed by `(symbol, method)`.
7. **`trade_paths()`** — defines `paths`. Per-bar cumulative-return Series.
   Feeds the aligned trade-paths chart and the entry-timing horizontal-bar chart.
8. **Distribution** — per-symbol return histograms with expectancy / median vlines.
9. **`SignalComparison`** — multi-variant harness. Includes the
   `BarBacktest(..., entry_offset=5)` *delayed-entry* hypothetical alongside
   the native run, plus any other MA-window variants you want to drop in.
10. **HTML Tearsheet** — `SignalTearsheet(...).save()` bundles every section above
    into one self-contained `.html`.

Column / flag definitions live in `hailmary.analytics` as `TRADE_STATS_DOCS`,
`TRADE_SUMMARY_DOCS`, `QUALITY_FLAG_DOCS`, `D5_STATS_DOCS`, `FILL_METHOD_DOCS`.
The HTML tearsheet's footnotes and the docs rendered below in each section are
both built from those same dicts — change a description in one place and both
surfaces update.

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.analytics import (
    SignalTradePerformance,
    TRADE_STATS_DOCS,
    TRADE_SUMMARY_DOCS,
    QUALITY_FLAG_DOCS,
    D5_STATS_DOCS,
    FILL_METHOD_DOCS,
    docs_markdown,
)
from hailmary.analytics.signal_comparison import SignalComparison
from hailmary.viz.signal_tearsheet import (
    SignalTearsheet,
    paths_fig,
    timing_fig,
    distribution_fig,
    quality_table_styler,
    trade_log_styler,
)

## 1. Pipeline & Setup

Fetch bars → run signal → run backtest → wrap in `SignalTradePerformance`.
This cell only builds `trade_perf`. Each section below defines its own
intermediate (`trades_df`, `ts_net`/`ts_con`, `paths`) at the point of first
use and threads it forward via the `trade_stats=` keyword.

In [2]:
signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start, end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-01-01")

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars = yahoo.get_bars(symbols, start=fetch_start, end=end, adjust=False)

signal_df  = signal.run(bars, trim_start=start)
bt_result  = BarBacktest().run(signal_df)
trade_perf = SignalTradePerformance(bt_result)

# Each section below defines its own intermediate at point of first use:
#   §2  trades_df = trade_perf.trade_stats()
#   §3  ts_net    = trade_perf.trade_summary("net",          trade_stats=trades_df)
#       ts_con    = trade_perf.trade_summary("conservative", trade_stats=trades_df)
#   §7  paths     = trade_perf.trade_paths(trade_stats=trades_df)

print(
    f"{bt_result.data.shape[0]:,} bars  "
    f"|  {bt_result.data.index.get_level_values('symbol').nunique()} symbols"
)

2026-05-10 19:18:12.349 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=de1ec4900549


2,193 bars  |  3 symbols


## 2. `trade_stats()`

The canonical per-trade scalar DataFrame: one row per entry-to-exit cycle.
Every other method on `SignalTradePerformance` derives from this — passing
the result back via `trade_stats=trades_df` avoids redoing the per-trade
groupby.

`trade_log_styler` shows the same data with green/red colouring on the
return columns and a red gradient on intra-trade drawdowns; trades shorter
than 5 bars carry a `(Nd)` muted suffix on the 5-day columns.

In [3]:
trades_df = trade_perf.trade_stats()

display(trade_log_styler(trades_df))
display(Markdown(docs_markdown(TRADE_STATS_DOCS, title="Columns")))

,label,sym,entry_date,exit_date,duration,d5_net,d5_con,d5_to_exit_net,d5_to_exit_con,final_net,final_con,max_dd_net,max_dd_con
0,BTC-USD T1,BTC-USD,2023-01-14,2023-08-18,217,+3.9%,-1.8%,+28.7%,+24.1%,+33.8%,+21.8%,-18.7%,-18.7%
1,BTC-USD T2,BTC-USD,2023-08-30,2023-08-31,2,-1.5%,-7.2%,+0.0%,+0.0%,-1.5%,-7.2%,-1.5%,-7.2%
2,BTC-USD T3,BTC-USD,2023-10-17,2024-01-01,77,+4.9%,+4.5%,+47.6%,+47.6%,+54.9%,+54.3%,-6.6%,-6.6%
3,ETH-USD T1,ETH-USD,2022-01-01,2022-01-08,8,-3.6%,-5.8%,-10.1%,-14.9%,-13.3%,-19.9%,-16.6%,-21.1%
4,ETH-USD T2,ETH-USD,2022-04-04,2022-04-06,3,-3.1%,-10.3%,+0.0%,+0.0%,-3.1%,-10.3%,-3.1%,-10.3%
5,ETH-USD T3,ETH-USD,2023-01-13,2023-08-18,218,+10.6%,+7.3%,+7.3%,+4.9%,+18.6%,+12.5%,-22.1%,-22.4%
6,ETH-USD T4,ETH-USD,2023-10-26,2023-10-28,3,-0.4%,-4.9%,+0.0%,+0.0%,-0.4%,-4.9%,-1.3%,-4.9%
7,ETH-USD T5,ETH-USD,2023-10-30,2024-01-01,64,+2.1%,+0.2%,+28.3%,+28.3%,+31.0%,+28.6%,-8.7%,-8.7%
8,SOL-USD T1,SOL-USD,2022-01-01,2022-01-20,20,-8.9%,-13.3%,-12.4%,-18.0%,-20.3%,-28.9%,-23.9%,-28.9%
9,SOL-USD T2,SOL-USD,2023-02-21,2023-02-22,2,-4.7%,-11.5%,+0.0%,+0.0%,-4.7%,-11.5%,-4.7%,-11.5%


**Columns**

- **Label** — Display label like 'BTC-USD T3' (T-numbered per symbol).
- **Symbol** — Symbol the trade was on.
- **Entry** — Bar timestamp at entry.
- **Exit** — Bar timestamp at exit.
- **Duration** — Number of bars the trade was held.
- **Return (Net)** — Compound net return (after fills + costs).
- **Return (MTC)** — Compound MTC return (no fill modelling).
- **Return (Conservative)** — Compound return, worst-fill assumption.
- **Max DD (Net)** — Worst peak-to-trough (≤ 0).
- **Max DD (MTC)** — Worst peak-to-trough, MTC fill.
- **Max DD (Conservative)** — Worst peak-to-trough, conservative fill.
- **5d (Net)** — Net return over first 5 bars (or full trade if shorter).
- **5d (MTC)** — MTC return over first 5 bars.
- **5d (Conservative)** — Conservative return over first 5 bars.
- **5d → Exit (Net)** — Net return bar 5 → close (0 if duration ≤ 5).
- **5d → Exit (MTC)** — MTC return bar 5 → close.
- **5d → Exit (Conservative)** — Conservative return bar 5 → close.

## 3. `trade_summary(method)`

Per-symbol aggregation of `trades_df`: collapses every trade for one
symbol into edge metrics — win rate, expectancy, profit factor, skew,
intra-trade DD. Returns one DataFrame per fill method.

In [4]:
ts_net = trade_perf.trade_summary("net",          trade_stats=trades_df)
ts_con = trade_perf.trade_summary("conservative", trade_stats=trades_df)

display(Markdown("**`trade_summary('net')`**"))
display(ts_net)
display(Markdown("**`trade_summary('conservative')`**"))
display(ts_con)
display(Markdown(docs_markdown(TRADE_SUMMARY_DOCS, title="Columns")))

**`trade_summary('net')`**

,n_trades,win_rate,avg_win,avg_loss,expectancy,expectancy_ex_top,median_return,profit_factor,max_win,max_loss,skewness,avg_duration,avg_intra_drawdown,max_intra_drawdown
symbol,,,,,,,,,,,,,,
BTC-USD,3.0,0.666667,0.443159,-0.015298,0.290340,0.161245,0.337788,57.936784,0.548530,-0.015298,-0.728671,98.666667,-0.089532,-0.186954
ETH-USD,5.0,0.400000,0.248154,-0.056161,0.065565,0.004442,-0.004139,2.945756,0.310059,-0.132918,0.531511,59.200000,-0.103776,-0.221455
SOL-USD,9.0,0.111111,4.119128,-0.069924,0.395526,-0.069924,-0.032464,7.363580,4.119128,-0.202685,2.986420,24.777778,-0.134164,-0.242070


**`trade_summary('conservative')`**

,n_trades,win_rate,avg_win,avg_loss,expectancy,expectancy_ex_top,median_return,profit_factor,max_win,max_loss,skewness,avg_duration,avg_intra_drawdown,max_intra_drawdown
symbol,,,,,,,,,,,,,,
BTC-USD,3.0,0.666667,0.380636,-0.072306,0.229655,0.072833,0.217971,10.528452,0.543300,-0.072306,0.170476,98.666667,-0.108479,-0.186954
ETH-USD,5.0,0.400000,0.205664,-0.116927,0.012109,-0.056352,-0.049144,1.172603,0.285952,-0.198688,0.628358,59.200000,-0.134777,-0.224091
SOL-USD,9.0,0.111111,3.488793,-0.129523,0.272512,-0.129523,-0.103125,3.366973,3.488793,-0.289381,2.975502,24.777778,-0.171649,-0.289381


**Columns**

- **N Trades** — Number of entry-to-exit cycles.
- **Win Rate** — Fraction of trades that closed positive.
- **Avg Win** — Mean return across winning trades.
- **Avg Loss** — Mean return across losing trades (negative).
- **Expectancy** — win_rate × avg_win + (1 − win_rate) × avg_loss.
- **Exp ex-Top** — Expectancy after dropping the best trade.
- **Median** — 50th-pct return; Median ≪ Expectancy = right-skewed.
- **Profit Factor** — Σ wins / |Σ losses|; > 1 earns more than it loses.
- **Max Win** — Best single-trade return.
- **Max Loss** — Worst single-trade return.
- **Skewness** — > +1: rare large wins; < −1: rare large losses.
- **Avg Duration** — Mean bars held per trade.
- **Avg DD** — Mean of per-trade max intra-drawdowns (≤ 0).
- **Worst DD** — Worst per-trade intra-drawdown observed (≤ 0).

## 4. `quality_flags(row)`

Boolean warning flags computed from one row of a `trade_summary`. The
cross-fill flags (`edge_reversed`, `fill_halves_edge`, `median_flips`)
only fire when a non-net method is compared to net via the optional
`net_row` keyword.

In [ ]:
sym = "BTC-USD"

flags_net = SignalTradePerformance.quality_flags(ts_net.loc[sym])
flags_con = SignalTradePerformance.quality_flags(
    ts_con.loc[sym],
    net_row=ts_net.loc[sym],
)

display(Markdown(f"**Net flags ({sym})**"))
display(pd.Series(flags_net))
display(Markdown(f"**Conservative flags ({sym}) — vs net**"))
display(pd.Series(flags_con))
display(Markdown(docs_markdown(QUALITY_FLAG_DOCS, title="Flags")))

## 5. `d5_stats(symbol, method)`

Entry-timing breakdown for one symbol: how often does a trade work by
bar 5 (`wr_d5`), and how often does the post-bar-5 tail add return
(`wr_tail`)?  `wr_d5` filters out trades that didn't reach bar 5;
`wr_tail` filters out trades ≤ 5 bars.

In [ ]:
d5_rows = [
    {"symbol": s, **SignalTradePerformance.d5_stats(s, "net", trade_stats=trades_df)}
    for s in trades_df["symbol"].unique()
]
display(Markdown("**`d5_stats(symbol, 'net')`** — across all symbols"))
display(pd.DataFrame(d5_rows).set_index("symbol"))
display(Markdown(docs_markdown(D5_STATS_DOCS, title="Fields")))

## 6. `quality_table(methods)`

The joined view: `trade_summary` + `d5_stats` + `quality_flags` per
`(symbol, method)` pair. `quality_table_styler` formats it with
diverging green/red on the edge metrics, descending-red on drawdowns,
and the raw `flags` dict rendered to compact text.

In [ ]:
quality_table_styler(trade_perf.quality_table(trade_stats=trades_df))

## 7. `trade_paths()`

Per-bar cumulative-return Series for every trade (one Series per fill
method, starting at 0.0 on the entry bar). Carries the scalar fields
(`final_*`, `d5_*`, `max_dd_*`) sourced from `trades_df` so chart hovers
don't have to look them up separately.

- `paths_fig(paths, method)` — every trade pivoted to *day 0 = entry*,
  win/loss-coloured, with mean / median / ±1σ overlays.
- `timing_fig(paths, method)` — horizontal stacked bar: first-5-bars
  return vs 5d→exit return per individual trade.

In [ ]:
paths = trade_perf.trade_paths(trade_stats=trades_df)

paths_fig(paths,  method="net").show()
paths_fig(paths,  method="conservative").show()
timing_fig(paths, method="net").show()
timing_fig(paths, method="conservative").show()

## 8. Return Distribution

Per-symbol histogram of trade returns — green bars are winners, red are
losers. Vertical lines mark the **expectancy** (solid) and the **median**
(purple dashed) for the chosen fill method. Expectancy ≫ median means a
few large winners are inflating the mean — the same outlier-driven edge
that the *Exp ex-Top* and *top trade outlier* flags surface in §6.

In [ ]:
distribution_fig(trades_df, ts_net, ts_con, method="net").show()
distribution_fig(trades_df, ts_net, ts_con, method="conservative").show()

## 9. `SignalComparison`

Drop multiple `BarBacktestResult` variants into the `variants` dict and
re-run — `SignalComparison(variants).tearsheet()` produces a three-panel
side-by-side view: pooled-quality table per variant, per-symbol expectancy
heatmap, and per-symbol return-distribution boxplots. Use this to rank
signal variants without bouncing between standalone tearsheets.

The cell below shows the **delayed-entry hypothetical** as a variant:
`BarBacktest().run(signal_df, entry_offset=5)` only takes a trade if the
signal still holds 5 bars after the original flip — the delay is an
*execution* parameter, not a separate signal class. Compare its expectancy
and trade count against `MA-200 Base` to see whether waiting for
confirmation improves the edge or just costs you trades.

In [ ]:
bt_delayed_5 = BarBacktest().run(signal_df, entry_offset=5)

variants = {
    "MA-200 Base":       bt_result,
    "MA-200 Delayed-5":  bt_delayed_5,
    # Add more here as you iterate. Each value is a BarBacktestResult.
    # e.g. BarBacktest().run(TrendSignal(ma_window=100).run(bars))
}

SignalComparison(variants).tearsheet().show()

## 10. Standalone HTML Tearsheets

`SignalTearsheet` bundles every section above (Per-Trade Quality → Entry
Timing → Trade Log → Aligned Trade Paths → Return Distribution) into one
self-contained `.html` file with sticky navigation and symbol/fill
filters — viewable in any browser without Jupyter.

Footnotes in the HTML's quality table are built from the same `*_DOCS`
dicts shown above — single source of truth across both surfaces.

The cell below saves **two** tearsheets, one per `BarBacktestResult` variant:

- `ma200_native.html` — the native run (`bt_result`).
- `ma200_delayed_5.html` — the `entry_offset=5` run (`bt_delayed_5` from §9).

Each tearsheet's `d5` / `5d→Exit` / timing surfaces describe the realized
trades of *its own* variant: native → first-5 bars from the original signal
flip; delayed → first-5 bars after the 5-bar confirmation hold. Open both
side-by-side in the browser to compare. Only the native one auto-opens to
keep the tab count reasonable.

In [ ]:
reports = Path("../../reports")

native_path = SignalTearsheet(
    trade_perf,
    title="MA-200 Trend Signal — BTC/ETH/SOL 2022–2024 (Native)",
).save(reports / "ma200_native.html", open=True)

delayed_path = SignalTearsheet(
    SignalTradePerformance(bt_delayed_5),
    title="MA-200 Trend Signal — BTC/ETH/SOL 2022–2024 (Delayed-5)",
).save(reports / "ma200_delayed_5.html", open=True)

print(f"Native  → {native_path}")
print(f"Delayed → {delayed_path}")